## Read Cross Validation Results

In [1]:
import numpy as np 
import pandas as pd
import altair as alt
from os import listdir
from os.path import isfile, join
from utils.util import read_results, read_eval_result

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_rows', 500)

In [2]:
#read the cross validation results
#mypath = "outputs/crossval"
mypath = "outputs/repeatedcrossval"
files = [f for f in listdir(mypath) if isfile(join(mypath, f))]

final_df = pd.DataFrame()

for file in files:
    if file.endswith('.xlsx'):
        filename = join(mypath,file)
        df = read_results(filename)
        final_df = pd.concat([final_df,df])

#filter model 
final_df = final_df[final_df['model'].isin(['SVM','RandomForest','LogisticRegression','NaiveBayes','DecisionTree'])]

#numbering the evaluation metrics as each metric has as many values as the number of n_fold cross validation (n=5)
eval_metrics = ['Accuracy','MCC','F_1','F_beta_0.5','F_beta_2','Precision','Recall']

for metric in eval_metrics:
    score_columns = final_df.filter(regex=metric).columns
    final_df['mean_{}'.format(metric)] = final_df[score_columns].mean(axis=1)
    final_df['std_{}'.format(metric)] = final_df[score_columns].std(axis=1)
    final_df['{}'.format(metric)] = final_df.apply(lambda x: '{:.3f}±{:.3f}'.format(x['mean_{}'.format(metric)],x['std_{}'.format(metric)]), axis=1)

#final_df = final_df[final_df['fold']=='StratifiedGroupKFold']
final_df.rename(columns={'mean_F_1':'mean_F1','F_1':'F1'}, inplace=True)

In [3]:
#rank the RCKmer and Kmer based on F1 score per dataset
kmer_family = final_df[(final_df.representation.str.startswith('Kmer'))].copy()
kmer_family['group_rank'] = kmer_family.groupby(['dataset','fold'])['mean_F1'].rank(method="first", ascending=False)

rckmer_family = final_df[(final_df.representation.str.startswith('RCKmer'))].copy()
rckmer_family['group_rank'] = rckmer_family.groupby(['dataset','fold'])['mean_F1'].rank(method="first", ascending=False)

kmer_table = pd.concat([kmer_family,rckmer_family])

In [4]:
#assign the rank of RCKmer and Kmer to the original table
df_merged = pd.merge(final_df, kmer_table[['dataset','representation','fold','group_rank']], on=['dataset','representation','fold'], how='left')
df_merged = df_merged.fillna(0.0)

In [5]:
#select the best RCKmer and Kmer only
sub_final_df = df_merged[df_merged['group_rank']<=1.0].copy()
sub_final_df['rank'] = sub_final_df.groupby(['dataset','representation','fold'])['mean_F1'].rank(method="first", ascending=False)
sub_final_df = sub_final_df[sub_final_df['rank']==1]

#show the top 5 performing representations along with the corresponding model sorted by F_1 score in descending order
sub_df_stratified_group_top_n = sub_final_df.groupby('dataset').apply(pd.DataFrame.nlargest, n=5, columns=['mean_F1'])
sub_df_stratified_group_top_n[['model','representation','F1','F_beta_0.5','F_beta_2','MCC','Accuracy','Precision','Recall']]#,'data_split_1', 'data_split_2','data_split_3','data_split_4','data_split_5' ]]

model  representation           F1  \
dataset                                                              
benbow       1049                 SVM        RCKmer-7  0.928±0.023   
             1014                 SVM          Kmer-7  0.909±0.026   
             1143                 SVM   Z_curve_48bit  0.880±0.039   
             350         RandomForest          PseKNC  0.878±0.037   
             1141                 SVM  Z_curve_144bit  0.874±0.030   
gicluster    2284                 SVM          Kmer-6  0.627±0.136   
             2324                 SVM        RCKmer-7  0.621±0.140   
             244   LogisticRegression        Mismatch  0.577±0.073   
             456                  SVM  Z_curve_144bit  0.576±0.099   
             246   LogisticRegression     Subsequence  0.571±0.095   
islandpick   1914                 SVM        RCKmer-7  0.893±0.057   
             1874                 SVM          Kmer-6  0.880±0.046   
             378         RandomForest   Z_curve_48bit  0.857±0.050   
             683         RandomForest         PseEIIP  0.847±0.052   
             687         RandomForest            TPCP  0.842±0.052   
islandviewer 174                  SVM        RCKmer-7  0.875±0.016   
             2499                 SVM          Kmer-7  0.869±0.017   
             228         RandomForest     Subsequence  0.838±0.024   
             428         RandomForest   Z_curve_48bit  0.821±0.023   
             567         RandomForest            TPCP  0.819±0.016   
rvm          1483        RandomForest        RCKmer-5  0.806±0.110   
             403         RandomForest   Z_curve_48bit  0.791±0.087   
             1448        RandomForest          Kmer-5  0.790±0.136   
             1203        RandomForest        PCPseTNC  0.773±0.095   
             1206        RandomForest        SCPseDNC  0.765±0.096   

                    F_beta_0.5     F_beta_2          MCC     Accuracy  \
dataset                                                                 
benbow       1049  0.923±0.032  0.933±0.024  0.837±0.048  0.919±0.024   
             1014  0.900±0.040  0.920±0.024  0.791±0.061  0.897±0.030   
             1143  0.875±0.054  0.886±0.036  0.727±0.079  0.864±0.042   
             350   0.859±0.057  0.899±0.020  0.717±0.066  0.860±0.036   
             1141  0.868±0.049  0.882±0.020  0.714±0.050  0.859±0.028   
gicluster    2284  0.619±0.179  0.665±0.137  0.497±0.197  0.760±0.141   
             2324  0.624±0.177  0.651±0.151  0.499±0.199  0.768±0.144   
             244   0.555±0.118  0.627±0.095  0.418±0.132  0.729±0.112   
             456   0.565±0.145  0.614±0.110  0.404±0.184  0.723±0.134   
             246   0.536±0.127  0.629±0.096  0.377±0.183  0.705±0.147   
islandpick   1914  0.881±0.087  0.909±0.032  0.832±0.087  0.918±0.050   
             1874  0.871±0.075  0.892±0.026  0.812±0.074  0.909±0.039   
             378   0.842±0.074  0.874±0.030  0.771±0.090  0.890±0.049   
             683   0.833±0.083  0.865±0.034  0.761±0.082  0.884±0.045   
             687   0.835±0.077  0.853±0.040  0.751±0.089  0.882±0.049   
islandviewer 174   0.868±0.028  0.884±0.016  0.748±0.036  0.873±0.019   
             2499  0.859±0.024  0.878±0.017  0.733±0.036  0.866±0.019   
             228   0.848±0.031  0.829±0.024  0.684±0.050  0.841±0.025   
             428   0.804±0.032  0.841±0.029  0.632±0.050  0.814±0.027   
             567   0.800±0.024  0.840±0.022  0.626±0.036  0.811±0.019   
rvm          1483  0.808±0.099  0.808±0.130  0.630±0.208  0.811±0.105   
             403   0.786±0.082  0.800±0.108  0.599±0.164  0.795±0.082   
             1448  0.788±0.110  0.797±0.166  0.608±0.220  0.799±0.111   
             1203  0.774±0.087  0.775±0.113  0.563±0.175  0.779±0.087   
             1206  0.763±0.076  0.771±0.123  0.550±0.168  0.771±0.084   

                     Precision       Recall  
dataset                                      
benbow       1049  0.921±0.041  0.936±0.030  
             1014  0.894±0.051 

In [17]:
top_benbow_pairs = sub_df_stratified_group_top_n[sub_df_stratified_group_top_n['dataset']=='gicluster'][['representation','model']].values
top_benbow_pairs = [pair[0]+'/'+pair[1] for pair in top_benbow_pairs]
top_benbow_pairs

['Kmer-6/SVM',
 'RCKmer-7/SVM',
 'Mismatch/LogisticRegression',
 'Z_curve_144bit/SVM',
 'Subsequence/LogisticRegression',
 'CKSNAP/SVM',
 'SCPseDNC/RandomForest',
 'Moran/RandomForest',
 'Z_curve_48bit/SVM',
 'Geary/NaiveBayes']

In [9]:
final_df[final_df['dataset']=='islandviewer'].groupby('representation').apply(pd.DataFrame.nlargest, n=5, columns=['mean_F1'])[['model','F1','F_beta_0.5','F_beta_2','MCC','Accuracy','Precision','Recall']]

model           F1   F_beta_0.5     F_beta_2  \
representation                                                                 
ASDC           36        RandomForest  0.785±0.030  0.766±0.032  0.806±0.036   
               42                 SVM  0.754±0.032  0.755±0.032  0.755±0.043   
               12        DecisionTree  0.709±0.025  0.694±0.032  0.724±0.023   
               24  LogisticRegression  0.652±0.036  0.657±0.026  0.650±0.061   
               30          NaiveBayes  0.646±0.051  0.681±0.036  0.617±0.075   
CKSNAP         37        RandomForest  0.810±0.025  0.791±0.030  0.832±0.032   
               43                 SVM  0.795±0.025  0.786±0.033  0.806±0.031   
               13        DecisionTree  0.728±0.029  0.720±0.030  0.737±0.033   
               25  LogisticRegression  0.661±0.047  0.682±0.036  0.646±0.081   
               31          NaiveBayes  0.645±0.044  0.677±0.033  0.619±0.062   
DAC            72        RandomForest  0.801±0.023  0.787±0.031  0.817±0.023   
               84                 SVM  0.742±0.033  0.744±0.034  0.741±0.042   
               24        DecisionTree  0.733±0.024  0.720±0.025  0.748±0.026   
               48  LogisticRegression  0.713±0.038  0.727±0.044  0.701±0.047   
               60          NaiveBayes  0.707±0.025  0.722±0.033  0.694±0.035   
DACC           73        RandomForest  0.794±0.032  0.780±0.042  0.810±0.033   
               85                 SVM  0.741±0.024  0.745±0.033  0.738±0.038   
               25        DecisionTree  0.723±0.020  0.714±0.022  0.734±0.023   
               49  LogisticRegression  0.720±0.030  0.736±0.038  0.707±0.045   
               61          NaiveBayes  0.691±0.029  0.704±0.032  0.679±0.047   
DCC            74        RandomForest  0.791±0.024  0.777±0.033  0.806±0.031   
               86                 SVM  0.734±0.026  0.741±0.033  0.728±0.035   
               50  LogisticRegression  0.715±0.036  0.733±0.041  0.699±0.047   
               26        DecisionTree  0.706±0.028  0.699±0.030  0.713±0.029   
               62          NaiveBayes  0.685±0.028  0.699±0.027  0.673±0.045   
DPCP           75        RandomForest  0.784±0.022  0.765±0.032  0.805±0.025   
               27        DecisionTree  0.716±0.021  0.701±0.026  0.731±0.018   
               87                 SVM  0.704±0.039  0.724±0.031  0.686±0.053   
               51  LogisticRegression  0.684±0.039  0.682±0.033  0.689±0.063   
               63          NaiveBayes  0.645±0.057  0.681±0.038  0.616±0.078   
GC             26  LogisticRegression  0.613±0.038  0.621±0.029  0.608±0.064   
               32          NaiveBayes  0.611±0.036  0.636±0.021  0.591±0.060   
               44                 SVM  0.586±0.030  0.643±0.030  0.539±0.038   
               14        DecisionTree  0.564±0.021  0.563±0.013  0.565±0.032   
               38        RandomForest  0.564±0.022  0.562±0.018  0.565±0.028   
Geary          76        RandomForest  0.793±0.026  0.776±0.035  0.811±0.028   
               88                 SVM  0.731±0.023  0.746±0.027  0.719±0.037   
               28        DecisionTree  0.729±0.030  0.718±0.033  0.741±0.031   
               52  LogisticRegression  0.711±0.031  0.729±0.032  0.695±0.046   
               64          NaiveBayes  0.702±0.028  0.720±0.035  0.687±0.043   
MMI            42        RandomForest  0.799±0.027  0.782±0.033  0.817±0.031   
               49                 SVM  0.777±0.022  0.771±0.025  0.784±0.030   
               28  LogisticRegression  0.732±0.026  0.745±0.029  0.721±0.042   
               14        DecisionTree  0.724±0.023  0.714±0.025  0.736±0.026   
               35          NaiveBayes  0.673±0.046  0.675±0.032  0.674±0.073   
Mismatch       39        RandomForest  0.787±0.022  0.818±0.024  0.760±0.032   
               27  LogisticRegression  0.752±0.026  0.763±0.027  0.743±0.039   
               15        DecisionTree  0.746±0.015  0.742±0.018  0.750±0.018   
               45               